Import neccessary libraries

In [1]:
import os
import json
import logging
import pandas as pd
import mlflow
import mlflow.sklearn
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

# 1. THIẾT LẬP LOGGING (Dùng force=True để Jupyter không bị lỗi khi chạy lại cell nhiều lần)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    force=True 
)
logger = logging.getLogger(__name__)

# 2. ĐƯỜNG DẪN DỮ LIỆU
CLEANED_DATA_PATH = Path("../data/processed/cleaned_data.csv")
CONFIG_PATH = Path("../artifacts/best_model_config.json")
TARGET_COLUMN = "churn_status"

ModuleNotFoundError: No module named 'pkg_resources'

In [ ]:
mlflow.set_tracking_uri("http://127.0.0.1:5000/")
mlflow.set_experiment("Netflix_Model_Tuning")
mlflow.sklearn.autolog()

NameError: name 'mlflow' is not defined

In [ ]:
df = pd.read_csv(CLEANED_DATA_PATH)

: 

: 

: 

In [ ]:
train_set = df[df['data_split'] == 'train']
test_set = df[df['data_split'] == 'test']

X_train = train_set.drop(columns=['churn_status', 'data_split'])
y_train = train_set['churn_status']

X_test = test_set.drop(columns=['churn_status', 'data_split'])
y_test = test_set['churn_status']

: 

: 

: 

In [ ]:
def objective(trial):
    # Define the hyperparameter search space for Random Forest
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 300),
        "max_depth": trial.suggest_int("max_depth", 3, 20),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 10),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 5),
        "criterion": trial.suggest_categorical("criterion", ["gini", "entropy"]),
        "random_state": 42
    }

    # Each Optuna trial is logged as a nested run within MLflow
    with mlflow.start_run(nested=True):
        model = RandomForestClassifier(**params)
        model.fit(X_train, y_train)
        
        # Predictions and metric calculation
        y_pred = model.predict(X_test)
        f1 = f1_score(y_test, y_pred)
        accuracy = accuracy_score(y_test, y_pred)

        # Log parameters and metrics for this specific trial
        mlflow.log_params(params)
        mlflow.log_metric("f1_score", f1)
        mlflow.log_metric("accuracy", accuracy)
        
        return f1  # Optuna will maximize this value

# 3. Execute Hyperparameter Tuning with Optuna
with mlflow.start_run(run_name="RandomForest_Optuna_Optimization"):
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=30) 

    # Log the best trial parameters and results to the parent run
    mlflow.log_params(study.best_params)
    mlflow.log_metric("best_f1_score", study.best_value)

    # Re-train the final model using the best parameters and log it to MLflow
    best_model = RandomForestClassifier(**study.best_params, random_state=42)
    best_model.fit(X_train, y_train)
    mlflow.sklearn.log_model(best_model, "best_rf_model")

    print(f"Tuning completed. Best F1 Score: {study.best_value}")
    print(f"Best Parameters: {study.best_params}")

: 

: 

: 